In [46]:
from openai import OpenAI
from dotenv import load_dotenv
import minsearch
import requests
import json
import os

load_dotenv()

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# Load and index documents

documents = [
    {
        "id": "1",
        "course": "data-engineering-zoomcamp",
        "section": "General Course-Related Questions",
        "question": "Can I still join the course after the start date?",
        "answer": "Yes, you can still join. There is no penalty for joining late. You won't be able to submit some of the early homeworks, but you can still take part in the course."
    },
    {
        "id": "2",
        "course": "data-engineering-zoomcamp",
        "section": "General Course-Related Questions",
        "question": "What are the prerequisites for this course?",
        "answer": "You need to know Python, SQL, and have some experience with the command line. Docker and cloud experience is helpful but not required."
    },
    {
        "id": "3",
        "course": "data-engineering-zoomcamp",
        "section": "General Course-Related Questions",
        "question": "How do I get a certificate?",
        "answer": "To get a certificate you need to complete the capstone project and get it reviewed by peers."
    },
    {
        "id": "4",
        "course": "data-engineering-zoomcamp",
        "section": "Workshop: dbt",
        "question": "How do I install dbt?",
        "answer": "pip install dbt-core dbt-bigquery. You can also use dbt Cloud which has a free tier."
    },
    {
        "id": "5",
        "course": "data-engineering-zoomcamp",
        "section": "Workshop: Docker",
        "question": "How do I run a container in Docker?",
        "answer": "Use docker run <image-name>. Add -it for interactive mode and -d for detached mode."
    },
    {
        "id": "6",
        "course": "llm-zoomcamp",
        "section": "General Course-Related Questions",
        "question": "I just discovered the course. Can I still join?",
        "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we are still accepting submissions."
    },
    {
        "id": "7",
        "course": "llm-zoomcamp",
        "section": "General Course-Related Questions",
        "question": "What LLM providers can I use?",
        "answer": "You can use OpenAI, Groq, Google Gemini, Ollama, or any OpenAI-compatible provider."
    },
    {
        "id": "8",
        "course": "llm-zoomcamp",
        "section": "Module 1",
        "question": "What is RAG?",
        "answer": "RAG stands for Retrieval Augmented Generation. It combines search with LLM generation to answer questions based on your own data."
    }
]

print(f"Total documents: {len(documents)}")

Total documents: 8


In [47]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "answer", "section"],
    keyword_fields=["course"]
)
index.fit(documents)
print("Index ready ")

Index ready 


In [48]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def search(query, course="data-engineering-zoomcamp"):
    boost = {'question': 3.0, 'section': 0.5}
    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=3
    )
    return results

In [49]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the FAQ knowledge base for relevant documents. Use this when you need to find information to answer a question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    },
                    "course": {
                        "type": "string",
                        "description": "The course to filter by e.g. data-engineering-zoomcamp or llm-zoomcamp",
                        "enum": ["data-engineering-zoomcamp", "llm-zoomcamp"]
                    }
                },
                "required": ["query", "course"]
            }
        }
    }
]

In [50]:
question = "Can I join the data engineering course after it started?"

messages = [
    {"role": "user", "content": question}
]

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=tools
)

print(response.choices[0].message)
print("\nFinish reason:", response.choices[0].finish_reason)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='5nde6xwrf', function=Function(arguments='{"course":"data-engineering-zoomcamp","query":"data engineering course join after started"}', name='search'), type='function')])

Finish reason: tool_calls


In [51]:
# extract the tool call
tool_call = response.choices[0].message.tool_calls[0]
function_name = tool_call.function.name
function_args = json.loads(tool_call.function.arguments)

print(f"Function: {function_name}")
print(f"Args: {function_args}")

# actually run the search
search_results = search(**function_args)
print(f"\nSearch returned {len(search_results)} results:")
for r in search_results:
    print(f"- {r['question']}")

Function: search
Args: {'course': 'data-engineering-zoomcamp', 'query': 'data engineering course join after started'}

Search returned 3 results:
- Can I still join the course after the start date?
- What are the prerequisites for this course?
- How do I get a certificate?


In [52]:
# build the full message history including tool result
messages = [
    {"role": "user", "content": question},
    response.choices[0].message,  # LLM's tool call request
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(search_results)
    }
]

# get final answer
final_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)

You can still join the data engineering course after it has started. There is no penalty for joining late, but you will be unable to submit some of the early homeworks. You need to know Python, SQL, and have some experience with the command line to enroll in the course.


In [53]:
def run_agent(question):
    messages = [{"role": "user", "content": question}]
    
    while True:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tools
        )
        
        message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        messages.append(message)
        
        if finish_reason == "stop":
            # LLM is done, return final answer
            return message.content
        
        if finish_reason == "tool_calls":
            # execute each tool call
            for tool_call in message.tool_calls:
                function_args = json.loads(tool_call.function.arguments)
                print(f"Searching: {function_args}")
                
                result = search(**function_args)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

In [54]:
answer = run_agent("What do I need to know before joining the data engineering course and how do I get a certificate?")
print(answer)

Searching: {'course': 'data-engineering-zoomcamp', 'query': 'data engineering course prerequisites and certificate'}
Searching: {'course': 'data-engineering-zoomcamp', 'query': 'data engineering course prerequisites'}
Searching: {'course': 'data-engineering-zoomcamp', 'query': 'data engineering capstone project'}
To join the data engineering course, you need to know Python, SQL, and have some experience with the command line. Docker and cloud experience is helpful but not required. You can join the course even after the start date, and there's no penalty for joining late. However, you won't be able to submit some of the early homeworks.

To get a certificate, you need to complete the capstone project and get it reviewed by peers.
